# Differentially Expressed Proteins
## UK Biobank (UKB) Cohort: Clinically Manifest ALS vs. Healthy Controls

**Author:** Ximing Ran

## Executive Summary

This notebook presents a comprehensive differential protein expression analysis comparing individuals with clinically manifest ALS to healthy controls in the UK Biobank cohort. Using a **1:1 greedy sex-and-age matched** design followed by a **limma** framework, we identify proteins that are significantly up- or down-regulated in ALS patients while accounting for demographic factors and genetic background.

**Analysis Pipeline:**
1. Load data
2. 1:1 Greedy matching (sex + age)
3. Quality Control
4. Exploratory analysis (spaghetti plots)
5. **limma** differential expression analysis for all proteins
6. Volcano plot
7. Pathway enrichment analysis (gprofiler2)

> **Why limma?** `limma` uses an empirical Bayes moderated *t*-statistic that borrows information across proteins to stabilise variance estimates — particularly valuable with small sample sizes (n = 44 here). This improves power and controls the FDR more accurately than protein-by-protein `lm()`.


## 1. Load Libraries

In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)
library(stringr)
library(limma)        # core DEA engine
library(ggrepel)
library(cowplot)
library(gprofiler2)
library(knitr)
library(splines)

set.seed(2024)
theme_set(theme_bw() + theme(legend.position = "bottom"))


## 2. Load Data

### 2.1 Protein Expression Data

The protein data is a **wide-format** RDS file where each row is a participant (`eid`) and each column is a protein (baseline measurement).


In [ ]:
# Load the baseline protein data (wide matrix: rows = samples, columns = proteins)
protein_data <- readRDS(here::here("data", "analysis_data", "ukb", "protein_data", "protein_data_baseline.rds"))

cat("Shape of the protein data:", dim(protein_data), "\n")
cat("First 10 column names:", head(colnames(protein_data), 10), "\n")


### 2.2 Visit / Phenotype Data

In [ ]:
# Load the visit info data
visit_info <- read.csv(here::here("data", "analysis_data", "ukb", "visit_info", "visit_info.csv"), row.names = 1)

cat("Shape of the visit info data:", dim(visit_info), "\n")


### 2.3 Study Population (Before Matching)

In [ ]:
# Group counts before matching
table(visit_info$Group)


In [ ]:
CTRL_AFF_raw <- visit_info %>% filter(Group %in% c("Healthy control", "Clinically manifest ALS"))

visit_summary_raw <- CTRL_AFF_raw %>%
  group_by(Group) %>%
  summarise(
    `Number of Person-Visit (#Visit)` = n(),
    `Number of Person (#N)`           = n_distinct(eid)
  ) %>%
  ungroup() %>%
  bind_rows(
    CTRL_AFF_raw %>%
      summarise(
        Group                            = "Total",
        `Number of Person-Visit (#Visit)` = n(),
        `Number of Person (#N)`           = n_distinct(eid)
      )
  )

knitr::kable(visit_summary_raw, caption = "Study Population Before Matching")


## 3. 1:1 Greedy Sex- and Age-Matched Cohort

To ensure comparability between ALS patients and healthy controls, we perform **1:1 greedy matching**:

- Match each Affected individual to the closest-age Control **of the same sex**
- Matching is done **without replacement** — each control is used at most once
- Individuals sorted by sex then age before matching to improve stability

This produces a balanced dataset that reduces confounding by age and sex in downstream analyses.


In [ ]:
# Split into Affected and Control
affected <- visit_info %>%
  filter(Group == "Clinically manifest ALS") %>%
  select(eid, Sex, CollAge, GenoGroup, Group, ALS, YrSinceDi, YrSinceCen) %>%
  distinct(eid, .keep_all = TRUE)

controls <- visit_info %>%
  filter(Group == "Healthy control") %>%
  select(eid, Sex, CollAge, GenoGroup, Group, ALS, YrSinceDi, YrSinceCen) %>%
  distinct(eid, .keep_all = TRUE)

cat("Affected:", nrow(affected), "| Controls:", nrow(controls), "\n")


In [ ]:
# 1:1 Greedy matching function: for each affected, pick closest-age control with same sex
match_1to1_greedy <- function(affected_df, controls_df) {
  controls_pool <- controls_df
  out <- vector("list", nrow(affected_df))

  # Sort by sex then age for greedy stability
  affected_df <- affected_df %>% arrange(Sex, CollAge)

  for (i in seq_len(nrow(affected_df))) {
    a <- affected_df[i, ]

    candidates <- controls_pool %>%
      filter(Sex == a$Sex) %>%
      mutate(age_diff = abs(CollAge - a$CollAge)) %>%
      arrange(age_diff, eid)

    if (nrow(candidates) == 0) {
      out[[i]] <- tibble(
        affected_eid = a$eid, affected_Sex = a$Sex, affected_Age = a$CollAge,
        control_eid  = NA_integer_, control_Age = NA_real_, age_diff = NA_real_
      )
      next
    }

    best <- candidates %>% slice(1)

    out[[i]] <- tibble(
      affected_eid = a$eid,
      affected_Sex = a$Sex,
      affected_Age = a$CollAge,
      control_eid  = best$eid,
      control_Age  = best$CollAge,
      age_diff     = best$age_diff
    )

    # Remove chosen control (without replacement)
    controls_pool <- controls_pool %>% filter(eid != best$eid)
  }

  bind_rows(out)
}

matched_pairs <- match_1to1_greedy(affected, controls)

# Quick QC summary
matched_pairs %>% summarise(
  n_affected      = n(),
  n_matched       = sum(!is.na(control_eid)),
  mean_age_diff   = mean(age_diff, na.rm = TRUE),
  median_age_diff = median(age_diff, na.rm = TRUE),
  max_age_diff    = max(age_diff, na.rm = TRUE)
)


In [ ]:
# Save matched pairs to CSV
write.csv(matched_pairs,
          here::here("data", "analysis_data", "ukb", "visit_info", "matched_pairs_affected_control.csv"),
          row.names = FALSE)

cat("Matched pairs saved.\n")


### 3.1 Build Matched Analysis Dataset

In [ ]:
# Collect all matched eids (affected + their matched controls)
matched_affected_eids <- matched_pairs %>% filter(!is.na(control_eid)) %>% pull(affected_eid)
matched_control_eids  <- matched_pairs %>% filter(!is.na(control_eid)) %>% pull(control_eid)
all_matched_eids      <- c(matched_affected_eids, matched_control_eids)

# Filter visit_info to matched eids only
CTRL_AFF <- visit_info %>%
  filter(eid %in% all_matched_eids, Group %in% c("Healthy control", "Clinically manifest ALS"))

# Summary after matching
visit_summary_matched <- CTRL_AFF %>%
  group_by(Group) %>%
  summarise(
    `Number of Person-Visit (#Visit)` = n(),
    `Number of Person (#N)`           = n_distinct(eid)
  ) %>%
  ungroup() %>%
  bind_rows(
    CTRL_AFF %>%
      summarise(
        Group                             = "Total",
        `Number of Person-Visit (#Visit)` = n(),
        `Number of Person (#N)`           = n_distinct(eid)
      )
  )

knitr::kable(visit_summary_matched, caption = "Study Population After 1:1 Matching")


In [ ]:
# Visualise age distribution before vs. after matching
par(mfrow = c(1, 2))

boxplot(CollAge ~ Group,
        data = visit_info %>% filter(Group %in% c("Healthy control", "Clinically manifest ALS")),
        main = "Age Distribution (Before Matching)",
        col  = c("lightblue", "salmon"),
        ylab = "Age", xlab = "Group")

boxplot(CollAge ~ Group,
        data = CTRL_AFF,
        main = "Age Distribution (After Matching)",
        col  = c("lightblue", "salmon"),
        ylab = "Age", xlab = "Group")

par(mfrow = c(1, 1))


In [ ]:
# Sex balance after matching
CTRL_AFF %>%
  group_by(Group, Sex) %>%
  summarise(N = n_distinct(eid), .groups = "drop") %>%
  tidyr::pivot_wider(names_from = Sex, values_from = N) %>%
  knitr::kable(caption = "Sex Balance After Matching")


## 4. Quality Control

### 4.1 Overview

We apply QC to the protein matrix:
1. Restrict to matched samples only
2. Remove proteins with **> 20% missing values** across the matched cohort


In [ ]:
# All protein columns (everything except eid)
all_proteins <- setdiff(colnames(protein_data), "eid")
cat("Total proteins before QC:", length(all_proteins), "\n")
cat("Total samples in protein_data:", nrow(protein_data), "\n")


### 4.2 Filter Protein Matrix to Matched Samples

In [ ]:
protein_matrix_filtered <- protein_data %>%
  filter(eid %in% all_matched_eids)

cat("Samples after matching filter:", nrow(protein_matrix_filtered), "\n")


### 4.3 Remove High-Missingness Proteins

In [ ]:
# Missingness rate per protein
missingness <- sapply(as.data.frame(protein_matrix_filtered)[, all_proteins, drop = FALSE],
                      function(x) mean(is.na(x)))
# Keep proteins with < 20% missing
keep_proteins <- names(missingness[missingness < 0.20])


In [ ]:
cat("Proteins retained (< 20% missing):", length(keep_proteins), "\n")
cat("Proteins removed:", length(all_proteins) - length(keep_proteins), "\n")

protein_matrix_qc <- protein_matrix_filtered %>%
  select(eid, all_of(keep_proteins))


In [ ]:
# QC Summary Table
summary_qc <- data.frame(
  Step = c("Original", "After Matching Filter", "After Missingness QC (< 20%)"),
  `Number of Samples`  = c(nrow(protein_data),
                            nrow(protein_matrix_filtered),
                            nrow(protein_matrix_qc)),
  `Number of Proteins` = c(length(all_proteins),
                            length(all_proteins),
                            length(keep_proteins)),
  check.names = FALSE
)

knitr::kable(summary_qc, caption = "QC Summary")


## 5. Prepare Analysis Data

Merge the QC-passed protein matrix with the matched visit/phenotype data.


In [ ]:
# Merge protein matrix (wide) with phenotype info
df_Int_input <- CTRL_AFF %>% mutate(eid = as.character(eid)) %>%
  left_join(protein_matrix_qc, by = "eid")

cat("Dimensions of merged data:", dim(df_Int_input), "\n")


## 6. Exploratory Analysis

Before formal modeling, we visualize protein expression vs. age for both groups using spaghetti plots:
- Individual points per sample
- Group-level **loess** and **linear** trend lines
- Reference lines: control 5th/95th percentiles and mean


In [ ]:
# Spaghetti plot function (adapted for UKB wide-format data)
plot_spaghetti_ukb <- function(data, protein, trend = "loess") {

  df_p <- data %>%
    select(eid, CollAge, Group, all_of(protein)) %>%
    rename(NPX = all_of(protein)) %>%
    filter(!is.na(NPX))

  df_ctrl  <- df_p %>% filter(Group == "Healthy control")
  top_95   <- quantile(df_ctrl$NPX, 0.95, na.rm = TRUE)
  bottom_5 <- quantile(df_ctrl$NPX, 0.05, na.rm = TRUE)
  mean_c   <- mean(df_ctrl$NPX, na.rm = TRUE)
  y_max    <- max(df_p$NPX, na.rm = TRUE)
  y_min    <- min(df_p$NPX, na.rm = TRUE)
  y_scale  <- y_max - y_min
  label_x  <- min(df_p$CollAge, na.rm = TRUE)

  p <- ggplot(df_p, aes(x = CollAge, y = NPX, color = Group)) +
    geom_point(alpha = 0.2) +
    geom_line(aes(group = eid), alpha = 0.15) +
    scale_color_manual(values = c("Healthy control" = "blue", "Clinically manifest ALS" = "red")) +
    coord_cartesian(ylim = c(y_min, y_max)) +
    geom_hline(yintercept = top_95,   linetype = "dashed", color = "black") +
    geom_hline(yintercept = bottom_5, linetype = "dashed", color = "black") +
    geom_hline(yintercept = mean_c,   linetype = "dotted", color = "black") +
    annotate("text", x = label_x, y = top_95   + 0.02 * y_scale,
             label = "CTRL Top 95%",  size = 3.5, hjust = 0) +
    annotate("text", x = label_x, y = bottom_5 + 0.02 * y_scale,
             label = "CTRL Bottom 5%", size = 3.5, hjust = 0) +
    annotate("text", x = label_x, y = mean_c   + 0.02 * y_scale,
             label = "CTRL Mean",     size = 3.5, hjust = 0) +
    labs(x = "Age at Collection", y = "Normalized Protein Level",
         title = sprintf("%s : %s trend", protein, trend)) +
    theme_classic(base_size = 13) +
    theme(plot.title    = element_text(hjust = 0.5, face = "bold"),
          legend.position = "right")

  if (trend == "loess") {
    p <- p + geom_smooth(method = "loess", aes(group = Group),
                         fill = "grey", alpha = 0.2, linewidth = 0.5,
                         se = TRUE, level = 0.95)
  } else {
    p <- p + geom_smooth(method = "lm", aes(group = Group),
                         fill = "grey", alpha = 0.2, linewidth = 0.5,
                         se = TRUE, level = 0.95)
  }
  p
}


In [ ]:
options(repr.plot.width = 16, repr.plot.height = 8)

demo_protein_list <- intersect(c("NEFL", "EDA2R", "MEGF10", "CA14", "FGFBP1", "ADGRG2"),
                                keep_proteins)
if (length(demo_protein_list) == 0) demo_protein_list <- keep_proteins[1:3]

cat("Plotting proteins:", paste(demo_protein_list, collapse = ", "), "\n")

for (protein in demo_protein_list) {
  p1 <- plot_spaghetti_ukb(df_Int_input, protein, trend = "loess") +
          theme(aspect.ratio = 1)
  p2 <- plot_spaghetti_ukb(df_Int_input, protein, trend = "lm") +
          theme(aspect.ratio = 1)
  print(plot_grid(p1, p2, nrow = 1))
}


## 7. Differential Expression Analysis: limma

### 7.1 Method Overview

`limma` (Linear Models for Microarray/Proteomics Data) fits the same design matrix as a classical linear model but adds **empirical Bayes moderation** of variance estimates. This is especially powerful for small-sample studies because:

- **Moderated *t*-statistics** shrink per-protein variances toward a global prior, reducing false positives from noisy low-variance proteins.
- Missing values are handled per-protein before building the expression matrix.
- Results are directly comparable to the `lm()` approach in notebook 1, but with improved sensitivity.

**Model (same covariates as notebook 1):**

$$Y_{i} = \beta_0 + \beta_1 X_{\text{ALS}} + \beta_2 X_{\text{Male}} + \beta_3 X_{\text{Age}} + \beta_4 X_{\text{Genotype}} + \varepsilon_{i}$$

P-values are **Benjamini-Hochberg adjusted** (FDR < 0.05).

### 7.2 Build Expression Matrix and Design Matrix


In [ ]:
# ── Step 1: Build samples × proteins expression matrix (proteins as rows, samples as cols)
# Use only samples that have complete phenotype data
pheno <- df_Int_input %>%
  select(eid, CollAge, Sex, GenoGroup, Group) %>%
  filter(!is.na(CollAge), !is.na(Sex), !is.na(GenoGroup)) %>%
  mutate(
    X_ALS      = ifelse(Group == "Clinically manifest ALS", 1L, 0L),
    X_Male     = ifelse(Sex == "Male", 1L, 0L),
    X_Age      = as.numeric(CollAge),
    X_Genotype = relevel(as.factor(GenoGroup), ref = "None identified")
  )

# Expression matrix: proteins × samples  (limma convention)
expr_wide <- df_Int_input %>%
  filter(eid %in% pheno$eid) %>%
  select(eid, all_of(keep_proteins))

# Reorder rows to match pheno
expr_wide <- expr_wide[match(pheno$eid, expr_wide$eid), ]

expr_mat <- t(as.matrix(expr_wide[, keep_proteins]))  # proteins × samples
colnames(expr_mat) <- pheno$eid

cat("Expression matrix dimensions (proteins × samples):", dim(expr_mat), "\n")
cat("Samples in pheno:", nrow(pheno), "\n")


### 7.3 Example: NEFL Protein (single-protein sanity check)

In [ ]:
protein_demo <- if ("NEFL" %in% keep_proteins) "NEFL" else keep_proteins[1]
cat("Demo protein:", protein_demo, "\n")

# Single-protein check using lmFit on one row
expr_demo <- expr_mat[protein_demo, , drop = FALSE]
ok_idx    <- !is.na(expr_demo[1, ])

design_demo <- model.matrix(
  ~ X_ALS + X_Male + X_Age + X_Genotype,
  data = pheno[ok_idx, ]
)

fit_demo  <- lmFit(expr_demo[, ok_idx, drop = FALSE], design_demo)
fit_demo  <- eBayes(fit_demo)

cat("\nlimma moderated t-test for", protein_demo, ":\n")
topTable(fit_demo, coef = "X_ALS", number = 1, adjust.method = "BH") %>%
  knitr::kable(caption = sprintf("limma result for %s", protein_demo))


### 7.4 Loop Over All Proteins with limma

We iterate over proteins (handling missing values per-protein), run `lmFit` + `eBayes`, extract the `X_ALS` coefficient, then apply BH correction across all proteins.

> **Note:** This loop can take several minutes. After the first run, comment it out and load results from the saved CSV.


In [ ]:
# ── RUN ONCE ─── comment out after first run and use read.csv below ─────────────
n_proteins <- length(keep_proteins)
cat("Fitting limma models for", n_proteins, "proteins...\n")

results_limma <- data.frame(
  Protein   = keep_proteins,
  Estimate  = rep(NA_real_, n_proteins),   # logFC / beta
  Std_Error = rep(NA_real_, n_proteins),
  t_value   = rep(NA_real_, n_proteins),
  Pr_t      = rep(NA_real_, n_proteins),
  stringsAsFactors = FALSE
)

for (i in seq_len(n_proteins)) {
  protein <- keep_proteins[i]
  expr_row <- expr_mat[protein, , drop = FALSE]

  # Drop samples with missing protein value
  ok_idx <- !is.na(expr_row[1, ])
  if (sum(ok_idx) < 5) next   # skip if too few observations

  pheno_sub  <- pheno[ok_idx, ]
  design_sub <- model.matrix(
    ~ X_ALS + X_Male + X_Age + X_Genotype,
    data = pheno_sub
  )

  tryCatch({
    fit   <- lmFit(expr_row[, ok_idx, drop = FALSE], design_sub)
    fit   <- eBayes(fit)
    coefs <- fit$coefficients[1, ]
    se    <- sqrt(fit$s2.post) * fit$stdev.unscaled[1, ]
    tstat <- fit$t[1, ]
    pval  <- fit$p.value[1, ]

    results_limma[i, "Estimate"]  <- coefs["X_ALS"]
    results_limma[i, "Std_Error"] <- se["X_ALS"]
    results_limma[i, "t_value"]   <- tstat["X_ALS"]
    results_limma[i, "Pr_t"]      <- pval["X_ALS"]
  }, error = function(e) {
    message(sprintf("Error for %s: %s", protein, e$message))
  })
}

# BH multiple testing correction
results_limma <- results_limma %>%
  mutate(
    padj          = p.adjust(Pr_t, method = "BH"),
    significant   = ifelse(padj < 0.05, "Significant", "Not-Significant"),
    diffexpressed = ifelse(Estimate > 0 & padj < 0.05, "UP",
                    ifelse(Estimate < 0 & padj < 0.05, "DOWN", "NO"))
  )

# Save results
write.csv(results_limma,
          "./Results/limma_ukb.csv",
          row.names = FALSE)

cat("\nDone!\n")
cat("Significant (FDR < 0.05):", sum(results_limma$significant == "Significant", na.rm = TRUE), "\n")
cat("  UP  :", sum(results_limma$diffexpressed == "UP",   na.rm = TRUE), "\n")
cat("  DOWN:", sum(results_limma$diffexpressed == "DOWN", na.rm = TRUE), "\n")


In [ ]:
# ── SUBSEQUENT RUNS: load pre-computed results ─────────────────────────────
# results_limma <- read.csv(here::here("Results", "limma_ukb.csv"))

cat("Total proteins tested:", nrow(results_limma), "\n")
cat("Significant (FDR < 0.05):", sum(results_limma$significant == "Significant", na.rm = TRUE), "\n")
cat("  UP  :", sum(results_limma$diffexpressed == "UP",   na.rm = TRUE), "\n")
cat("  DOWN:", sum(results_limma$diffexpressed == "DOWN", na.rm = TRUE), "\n")


### 7.5 Top Differentially Expressed Proteins

In [ ]:
# Show top 20 significant proteins by adjusted p-value
top_proteins <- results_limma %>%
  filter(significant == "Significant") %>%
  arrange(padj) %>%
  head(20) %>%
  mutate(across(where(is.numeric), ~ round(., 4)))

knitr::kable(top_proteins, caption = "Top 20 Differentially Expressed Proteins (limma, FDR < 0.05)")


## 8. Volcano Plot

The volcano plot visualizes effect size ($\hat{\beta}_1$) vs. statistical significance ($-\log_2$ adjusted p-value):

- **X-axis:** Effect Size (Beta1) — positive = higher in ALS
- **Y-axis:** $-\log_2(\text{FDR})$ — higher = more significant
- **Dashed line:** FDR = 0.05 threshold
- **Red:** Significantly up-regulated | **Blue:** Down-regulated | **Grey:** Not significant


In [ ]:
results_protein_plot <- results_limma %>% arrange(desc(padj))

num_up   <- sum(results_protein_plot$diffexpressed == "UP",   na.rm = TRUE)
num_down <- sum(results_protein_plot$diffexpressed == "DOWN", na.rm = TRUE)
max_fc   <- max(abs(results_protein_plot$Estimate), na.rm = TRUE)
max_p    <- max(-log2(results_protein_plot$padj),   na.rm = TRUE)

# Top 10 labelled proteins on each side
top_10_up <- results_protein_plot %>%
  filter(diffexpressed == "UP") %>%
  arrange(padj) %>%
  slice_head(n = 10)

top_10_down <- results_protein_plot %>%
  filter(diffexpressed == "DOWN") %>%
  arrange(padj) %>%
  slice_head(n = 10)

top_10_genes <- bind_rows(top_10_up, top_10_down)

p_volcano <- ggplot(results_protein_plot,
                    aes(Estimate, -log2(padj), color = diffexpressed)) +
  geom_point(alpha = 0.7, show.legend = FALSE) +
  scale_color_manual(values = c("DOWN" = "blue", "NO" = "grey", "UP" = "red")) +
  scale_x_continuous(limits = c(-max_fc - 0.1, max_fc + 0.1),
                     name   = "Effect Size (Beta1: ALS vs. Control)") +
  scale_y_continuous(limits = c(0, max_p + 0.5),
                     name   = "-log2(Adjusted P-value)") +
  geom_hline(yintercept = -log2(0.05), linetype = "dashed", color = "black") +
  geom_label_repel(data       = top_10_genes,
                   aes(label  = Protein, color = diffexpressed),
                   size       = 4,
                   max.overlaps = 50,
                   show.legend  = FALSE) +
  annotate("text",
           x = 0.01 * max_fc + 1, y = 0.85 * max_p,
           label = paste("UP:", num_up),   color = "red",  size = 6) +
  annotate("text",
           x = -0.01 * max_fc - 1, y = 0.85 * max_p,
           label = paste("DOWN:", num_down), color = "blue", size = 6) +
  theme_classic(base_size = 13) +
  theme(axis.title = element_text(size = 16, color = "black"),
        axis.text  = element_text(size = 14, color = "black")) +
  ggtitle("Differential Protein Expression: ALS vs. Control (UKB) — limma")

print(p_volcano)


## 9. Pathway Enrichment Analysis (g:Profiler)

We submit the significantly up- and down-regulated proteins separately to g:Profiler for over-representation analysis across GO, KEGG, and Reactome databases.


In [ ]:
# Significant proteins
sig_up   <- results_limma %>% filter(diffexpressed == "UP")   %>% pull(Protein)
sig_down <- results_limma %>% filter(diffexpressed == "DOWN") %>% pull(Protein)
bg_genes <- results_limma$Protein   # full tested set as background

cat("UP proteins:", length(sig_up), "\n")
cat("DOWN proteins:", length(sig_down), "\n")


In [ ]:
# g:Profiler enrichment — UP proteins
if (length(sig_up) > 0) {
  gp_up <- gost(
    query           = sig_up,
    organism        = "hsapiens",
    custom_bg       = bg_genes,
    correction_method = "fdr",
    sources         = c("GO:BP", "GO:MF", "KEGG", "REAC"),
    evcodes         = TRUE
  )
  if (!is.null(gp_up$result)) {
    cat("\nTop UP-regulated pathways:\n")
    gp_up$result %>%
      arrange(p_value) %>%
      select(source, term_name, p_value, term_size, intersection_size) %>%
      head(15) %>%
      mutate(p_value = round(p_value, 4)) %>%
      knitr::kable(caption = "g:Profiler: UP-regulated proteins")
  } else {
    cat("No significant pathways for UP proteins.\n")
  }
} else {
  cat("No UP proteins to test.\n")
}


In [ ]:
# g:Profiler enrichment — DOWN proteins
if (length(sig_down) > 0) {
  gp_down <- gost(
    query           = sig_down,
    organism        = "hsapiens",
    custom_bg       = bg_genes,
    correction_method = "fdr",
    sources         = c("GO:BP", "GO:MF", "KEGG", "REAC"),
    evcodes         = TRUE
  )
  if (!is.null(gp_down$result)) {
    cat("\nTop DOWN-regulated pathways:\n")
    gp_down$result %>%
      arrange(p_value) %>%
      select(source, term_name, p_value, term_size, intersection_size) %>%
      head(15) %>%
      mutate(p_value = round(p_value, 4)) %>%
      knitr::kable(caption = "g:Profiler: DOWN-regulated proteins")
  } else {
    cat("No significant pathways for DOWN proteins.\n")
  }
} else {
  cat("No DOWN proteins to test.\n")
}


## 10. Discussion

### Methodological Strengths
1. **1:1 matched design** minimizes confounding by age and sex
2. **Comprehensive covariate adjustment**: age, sex, genotype
3. **limma empirical Bayes moderation** stabilises variance estimates — especially valuable at n = 44
4. **BH correction** controls false discovery rate

### limma vs. lm() (notebook 1)
| Aspect | `lm()` (Notebook 1) | `limma` (Notebook 2) |
|--------|---------------------|----------------------|
| Variance estimation | Per-protein | Empirical Bayes pooled |
| Small-sample power | Lower | Higher |
| Computation | Slower (loop) | Fast (vectorised) |
| Missing values | Per-protein exclusion | Per-protein exclusion |
| FDR control | BH on raw p | BH on moderated p |

> Proteins detected by `limma` but not `lm()` are likely those with moderately high variance that benefit most from borrowing information across the proteome.
